## 5-6. Regularized Linear Models – Ridge, Lasso
### Regularized Linear Model - Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge  # 릿지 회귀 (L2 규제 선형 회귀)
from sklearn.model_selection import cross_val_score

# 보스턴 주택가격 데이터셋 다시 로드 및 DataFrame 변환
boston = load_boston()
bostonDF = pd.DataFrame(boston.data, columns=boston.feature_names)
bostonDF['PRICE'] = boston.target

# 타겟(y)과 피처(X) 분리
y_target = bostonDF['PRICE']
X_data = bostonDF.drop(['PRICE'], axis=1, inplace=False)

# Ridge 회귀 모델 생성 (alpha=10: 규제 강도)
# alpha가 클수록 회귀 계수를 작게 만들어 과적합 방지 (but 너무 크면 과소적합)
# 릿지는 L2 규제: 비용함수 = MSE + alpha * Σ(w²) → 회귀 계수의 제곱합에 페널티 부여
ridge = Ridge(alpha=10)

# 5-Fold 교차 검증으로 릿지 회귀 성능 평가
neg_mse_scores = cross_val_score(ridge, X_data, y_target, scoring="neg_mean_squared_error", cv=5)

# Negative MSE → RMSE로 변환
rmse_scores = np.sqrt(-1 * neg_mse_scores)
avg_rmse = np.mean(rmse_scores)

# 결과 출력: 일반 선형 회귀와 비교하여 RMSE가 개선되었는지 확인
print(' 5 folds 의 개별 Negative MSE scores: ', np.round(neg_mse_scores, 3))
print(' 5 folds 의 개별 RMSE scores : ', np.round(rmse_scores, 3))
print(' 5 folds 의 평균 RMSE : {0:.3f} '.format(avg_rmse))

**alpha값을 0 , 0.1 , 1 , 10 , 100 으로 변경하면서 RMSE 측정**

In [ ]:
# 다양한 alpha 값에 따른 릿지 회귀 성능 비교
# alpha=0: 규제 없음 (일반 선형 회귀와 동일)
# alpha가 증가할수록 규제가 강해져 회귀 계수가 작아짐
alphas = [0, 0.1, 1, 10, 100]

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    
    # 5-Fold 교차 검증으로 평균 RMSE 계산
    neg_mse_scores = cross_val_score(ridge, X_data, y_target, scoring="neg_mean_squared_error", cv=5)
    avg_rmse = np.mean(np.sqrt(-1 * neg_mse_scores))
    
    # alpha별 평균 RMSE 출력 → 최적의 alpha 값을 찾기 위한 비교
    # alpha=0 (규제 없음)보다 적절한 alpha에서 RMSE가 낮아지면 규제가 효과적임을 의미
    print('alpha {0} 일 때 5 folds 의 평균 RMSE : {1:.3f} '.format(alpha, avg_rmse))

**각 alpha에 따른 회귀 계수 값을 시각화. 각 alpha값 별로 plt.subplots로 맷플롯립 축 생성**

In [ ]:
# 1행 5열 서브플롯 생성 (각 alpha 값별로 하나의 막대 그래프)
fig, axs = plt.subplots(figsize=(18, 6), nrows=1, ncols=5)

# 각 alpha별 회귀 계수를 저장할 빈 DataFrame 생성
coeff_df = pd.DataFrame()

# 각 alpha 값에 대해 릿지 회귀 학습 후 회귀 계수 시각화
for pos, alpha in enumerate(alphas):
    ridge = Ridge(alpha=alpha)
    # 전체 데이터로 학습 (교차 검증이 아닌 회귀 계수 확인 목적)
    ridge.fit(X_data, y_target)
    
    # 학습된 회귀 계수를 Series로 변환 (인덱스: 피처명, 값: 회귀 계수)
    coeff = pd.Series(data=ridge.coef_, index=X_data.columns)
    colname = 'alpha:' + str(alpha)
    # DataFrame에 alpha별 회귀 계수를 컬럼으로 추가 (나중에 표로 비교하기 위해)
    coeff_df[colname] = coeff
    
    # 회귀 계수를 내림차순 정렬하여 막대 그래프로 시각화
    coeff = coeff.sort_values(ascending=False)
    axs[pos].set_title(colname)       # 그래프 제목: alpha 값
    axs[pos].set_xlim(-3, 6)          # x축 범위 고정 (alpha 간 비교 용이)
    # seaborn barplot: 가로 막대 그래프로 각 피처의 회귀 계수 표시
    # alpha가 커질수록 막대(회귀 계수)가 작아지는 것을 시각적으로 확인 가능
    sns.barplot(x=coeff.values, y=coeff.index, ax=axs[pos])

# 모든 서브플롯 표시
plt.show()

**alpha 값에 따른 컬럼별 회귀계수 출력**

In [ ]:
# alpha 값별 회귀 계수를 DataFrame 형태로 비교 출력
ridge_alphas = [0, 0.1, 1, 10, 100]

# 첫 번째 alpha(=0)의 회귀 계수 기준으로 내림차순 정렬
# alpha=0일 때가 규제 없는 선형 회귀이므로, 이를 기준으로 alpha 증가 시 계수 변화를 관찰
# alpha가 커질수록 큰 회귀 계수들이 줄어드는 것을 확인할 수 있음
sort_column = 'alpha:' + str(ridge_alphas[0])
coeff_df.sort_values(by=sort_column, ascending=False)

### 라쏘 회귀

In [ ]:
from sklearn.linear_model import Lasso, ElasticNet  # 라쏘(L1 규제), 엘라스틱넷(L1+L2 규제) 회귀

# 규제 선형 회귀 모델(Ridge, Lasso, ElasticNet)의 alpha별 성능을 평가하고 회귀 계수를 반환하는 범용 함수
# model_name: 모델 종류 ('Ridge', 'Lasso', 'ElasticNet')
# params: 평가할 alpha 값 리스트
# X_data_n, y_target_n: 피처와 타겟 데이터
# verbose: True이면 모델 이름 헤더 출력
# return_coeff: True이면 회귀 계수 DataFrame 반환
def get_linear_reg_eval(model_name, params=None, X_data_n=None, y_target_n=None, 
                        verbose=True, return_coeff=True):
    coeff_df = pd.DataFrame()  # alpha별 회귀 계수를 저장할 DataFrame
    if verbose: print('####### ', model_name, '#######')
    
    for param in params:
        # model_name에 따라 적절한 모델 객체 생성
        if model_name == 'Ridge': model = Ridge(alpha=param)
        elif model_name == 'Lasso': model = Lasso(alpha=param)
        # ElasticNet: l1_ratio=0.7 → L1 규제 70%, L2 규제 30% 혼합
        elif model_name == 'ElasticNet': model = ElasticNet(alpha=param, l1_ratio=0.7)
        
        # 5-Fold 교차 검증으로 Negative MSE 계산
        neg_mse_scores = cross_val_score(model, X_data_n, 
                                             y_target_n, scoring="neg_mean_squared_error", cv=5)
        # 평균 RMSE 계산 및 출력
        avg_rmse = np.mean(np.sqrt(-1 * neg_mse_scores))
        print('alpha {0}일 때 5 폴드 세트의 평균 RMSE: {1:.3f} '.format(param, avg_rmse))
        
        # cross_val_score는 평가 지표만 반환하고 모델 자체는 반환하지 않으므로
        # 회귀 계수를 확인하기 위해 전체 데이터로 다시 학습
        model.fit(X_data_n, y_target_n)
        
        if return_coeff:
            # 학습된 회귀 계수를 Series로 변환 (인덱스: 피처명)
            coeff = pd.Series(data=model.coef_, index=X_data_n.columns)
            colname = 'alpha:' + str(param)
            # DataFrame에 alpha별 회귀 계수를 컬럼으로 추가
            coeff_df[colname] = coeff
    
    return coeff_df  # alpha별 회귀 계수가 담긴 DataFrame 반환
# end of get_linear_regre_eval

In [ ]:
# 라쏘(Lasso) 회귀에 사용할 alpha 값들 정의
# 라쏘는 L1 규제: 비용함수 = MSE + alpha * Σ|w| → 회귀 계수의 절댓값 합에 페널티
# L1 규제의 특징: alpha가 커지면 일부 회귀 계수를 정확히 0으로 만듦 → 피처 선택 효과
lasso_alphas = [0.07, 0.1, 0.5, 1, 3]

# get_linear_reg_eval 함수를 호출하여 각 alpha별 RMSE와 회귀 계수 계산
coeff_lasso_df = get_linear_reg_eval('Lasso', params=lasso_alphas, X_data_n=X_data, y_target_n=y_target)

In [ ]:
# 라쏘 회귀의 alpha별 회귀 계수를 DataFrame으로 출력
# 첫 번째 alpha(=0.07) 기준 내림차순 정렬
# alpha가 커질수록 0이 되는 회귀 계수가 증가하는 것을 확인 가능
# → 라쏘의 피처 선택(feature selection) 효과: 중요하지 않은 피처의 계수를 0으로 만듦
sort_column = 'alpha:' + str(lasso_alphas[0])
coeff_lasso_df.sort_values(by=sort_column, ascending=False)

### 엘라스틱넷 회귀

In [ ]:
# 엘라스틱넷(ElasticNet) 회귀에 사용할 alpha 값들 정의
# 엘라스틱넷: L1(라쏘) + L2(릿지) 규제를 혼합한 모델
# 비용함수 = MSE + alpha * (l1_ratio * Σ|w| + (1-l1_ratio) * Σw²)
# l1_ratio=0.7: L1 규제 70% + L2 규제 30% 혼합 (get_linear_reg_eval 함수 내부에서 설정)
elastic_alphas = [0.07, 0.1, 0.5, 1, 3]

# 각 alpha별 RMSE 평가 및 회귀 계수 계산
coeff_elastic_df = get_linear_reg_eval('ElasticNet', params=elastic_alphas,
                                      X_data_n=X_data, y_target_n=y_target)

In [ ]:
# 엘라스틱넷 회귀의 alpha별 회귀 계수를 DataFrame으로 출력
# 첫 번째 alpha(=0.07) 기준 내림차순 정렬
# 라쏘와 유사하게 alpha가 커지면 일부 계수가 0이 되지만,
# L2 규제도 혼합되어 있어 라쏘보다 계수가 완전히 0이 되는 경우가 적음
sort_column = 'alpha:' + str(elastic_alphas[0])
coeff_elastic_df.sort_values(by=sort_column, ascending=False)

### 선형 회귀 모델을 위한 데이터 변환

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures

# 데이터 스케일링(변환) 함수: 다양한 전처리 방법을 적용
# method: 변환 방법 ('Standard', 'MinMax', 'Log', 'None')
# p_degree: 다항식 차수 (None이면 적용 안 함, 2 이상 지정 시 다항식 피처 추가)
# input_data: 변환할 입력 데이터
def get_scaled_data(method='None', p_degree=None, input_data=None):
    if method == 'Standard':
        # StandardScaler: 평균=0, 표준편차=1로 변환 (표준 정규 분포)
        # 각 피처를 (값 - 평균) / 표준편차로 변환
        scaled_data = StandardScaler().fit_transform(input_data)
    elif method == 'MinMax':
        # MinMaxScaler: 최솟값=0, 최댓값=1로 변환 (0~1 범위로 정규화)
        # 각 피처를 (값 - 최솟값) / (최댓값 - 최솟값)으로 변환
        scaled_data = MinMaxScaler().fit_transform(input_data)
    elif method == 'Log':
        # 로그 변환: np.log1p = log(1+x) → 왜도(skewness)가 큰 데이터를 정규분포에 가깝게 변환
        # log1p를 사용하는 이유: x=0일 때 log(0)=-∞ 방지
        scaled_data = np.log1p(input_data)
    else:
        # 변환 없이 원본 데이터 그대로 사용
        scaled_data = input_data

    if p_degree != None:
        # 다항식 피처 변환 추가: 기존 피처에 교차항과 거듭제곱항을 추가
        # include_bias=False: 상수항(1) 제외
        scaled_data = PolynomialFeatures(degree=p_degree, 
                                         include_bias=False).fit_transform(scaled_data)
    
    return scaled_data

In [ ]:
# Ridge 회귀에서 다양한 alpha 값 설정
alphas = [0.1, 1, 10, 100]

# 6가지 데이터 변환 방법 정의: (변환 방법, 다항식 차수)
# (None, None): 원본 데이터 그대로
# ('Standard', None): 표준 정규 분포 변환만
# ('Standard', 2): 표준 정규 분포 변환 + 2차 다항식 피처 추가
# ('MinMax', None): 최대/최소 정규화만
# ('MinMax', 2): 최대/최소 정규화 + 2차 다항식 피처 추가
# ('Log', None): 로그 변환만
scale_methods = [(None, None), ('Standard', None), ('Standard', 2), 
               ('MinMax', None), ('MinMax', 2), ('Log', None)]

for scale_method in scale_methods:
    # 각 변환 방법에 따라 피처 데이터를 변환
    X_data_scaled = get_scaled_data(method=scale_method[0], p_degree=scale_method[1], 
                                    input_data=X_data)
    # 변환 후 shape 출력: 다항식 피처 추가 시 컬럼 수가 크게 증가
    # 원본 (506, 13) → 2차 다항식 적용 시 (506, 104)
    print(X_data_scaled.shape, X_data.shape)
    print('\n## 변환 유형:{0}, Polynomial Degree:{1}'.format(scale_method[0], scale_method[1]))
    
    # 각 변환 방법별로 Ridge 회귀의 alpha에 따른 RMSE 출력
    # verbose=False: 모델명 헤더 출력 안 함, return_coeff=False: 회귀 계수 반환 안 함
    # → 어떤 변환 방법이 가장 좋은 성능을 내는지 비교
    get_linear_reg_eval('Ridge', params=alphas, X_data_n=X_data_scaled, 
                        y_target_n=y_target, verbose=False, return_coeff=False)